In [ ]:
import pandas as pd
from pathlib import Path
import json

LOADING DATA

In [ ]:
# checking if the dat exists
data_path = Path.cwd() / "SO_queries.json"
print(f"Data exist: {data_path.exists()}")
print(f"Check if is a file: {data_path.is_file()}")

In [ ]:
# extracting the SO queries

with open(data_path, 'r') as file:
    data = []
    for line in file:
        data.append(json.loads(line))
    
df = pd.DataFrame(data)
df = df.drop(columns=['question_title', 'answer_body'])
df.head()

In [ ]:
print(f"Null values check.")
df.isna().sum()

In [ ]:
# all topics
topics = df['topic'].unique()
topics

In [ ]:
# 3000 was the max amount of data pulled per topic, trying to see if there's undersampled classes

def topics_under_size(df, counts=1000):
    
    topic_counts = df.loc[:,'topic'].value_counts()
    topic_counts_filter = topic_counts < counts
    return topic_counts[topic_counts_filter].sort_values()

topics_under_size(df)

In [ ]:
# getting the minory class labels
df_dicts = {}
for column in topics_under_size(df).index:
   df_dicts[column] = df[df['topic'] == column]
   
df_dicts

In [ ]:
sample = df.sample(n=1)
sample.to_dict('records')[0]

new_df = df.copy()
pd.concat([new_df, pd.DataFrame([sample.to_dict('records')[0]])])

In [ ]:
import ollama

def generate_synth_data(df_examples : pd.DataFrame = None, topic_name= None, batch_gen=50, amount=500, model = 'gemma4:31b-cloud'):
    """
    Generates synthetic data for minority classes.
    """
    count = 0
    with open(Path.cwd() / f'{topic_name}.ndjson', 'a') as j:
        
        while count < amount:
            
            print(f'GENERATING "{topic_name}" samples... progress: {count} / {amount}')
        
            try:
                
                sample = df_examples.sample(n=5).to_dict('records')[0]
                
                response = ollama.chat(model=model,
                            messages=[{'role': 'system', 'content' : f'You are tasked with generating synthetic data in this exact format {{"topic": ..., "question_body" : ...}}. The user will provide examples and from that, you will generate an entirely new question_body under the SAME TOPIC. Everything should be natural sounding. Avoid adding backslashes to keys \\"topics\\" or \\"question_body"\\ is wrong. Use "" around the keys. Generate {batch_gen} copies separated by <SPLT>. Do not add whitespaces or new lines inside the keys.'},
                                    {'role': 'user', 'content' : f'{sample}'}],
                            think=False,
                            options={'temperature' : 0.8})
                
                content = response.message.content
                
                for line in content.split('<SPLT>'):
                    row = json.loads(line.strip())
                    j.write(json.dumps(row) + '\n')
                    j.flush()
                    
                count += batch_gen
            
            except Exception as e:
                print(f'Failed: {e}. Trying again...')

In [ ]:
# generating synthetic data to rebalance values
"""
for topic, topic_df in df_dicts.items():
amount = 1000 - len(topic_df)
generate_synth_data(topic_df, topic, batch_gen=50, amount=amount)
"""

In [ ]:
df.shape

In [ ]:
# we can fix some of these with mapping close topics together.
new_df = df.copy()
for topic, topic_df in df_dicts.items():
    
    json_df = pd.read_json(Path.cwd() / f'{topic}.ndjson', lines=1)
    new_df = pd.concat([json_df, new_df])
    
    
new_df.shape

In [ ]:
new_df['topic'].value_counts()

In [ ]:
# checking the number of samples per class
topics_under_size(new_df)

VISUALIZING DATA (KINDA)

In [ ]:

def random_sample_topics(df, number_per_topic=2):
        
    # randomly sample 
    random_samples = df.groupby('topic').sample(n=number_per_topic, random_state=42)

    return random_samples
    
random_sample_topics(new_df)

NOTE: Due to the nature of our data (heavily code-based), common cleaning techniques can serve more to hinder than help. According to the average sample, there are a lot of new lines. These are likely the things that need the most attention.

In [ ]:
import html

cleaned_df = new_df.copy()

# decode html code back into regular characters
cleaned_df = cleaned_df.apply(html.unescape)

# if the string pattern matches newline or carriage return (\r) ONE or MORE time, replace with space
cleaned_df = cleaned_df.replace(r'[\n\r]+', ' ', regex=True)

In [ ]:
random_sample_topics(cleaned_df)

In [ ]:
# cleaned df relable

cleaned_df.columns = {'label' : 'topic', 'content' : 'question_body'}
cleaned_df

In [ ]:
# saving to parquet
save_path = Path.cwd() / "train_ready.parquet"
cleaned_df.to_parquet(path=save_path)

if save_path.is_file():
    
    try:
        test = pd.read_parquet(save_path)
        
        print('If all zero, then file integrity is good.')
        print((test != cleaned_df).sum())
        
        print(f'Successfully saved: {save_path}')
        
    except Exception as e:
        print('Save Failed.', {e})
else:
    print('Save failed.')